In [ ]:
#| default_exp foundation

## Foundation

Small shared mechanics: CLI behavior, failure telemetry, cell parsing, validation, chapter navigation, and dynamic cell classification for notebook context.

This notebook is the shared basement of the project. It keeps the small private helpers that every public tool relies on: CLI behavior, safe error reporting, cell selection, cell parsing, semantic cell classification, and chapter addressing.

Most functions here are intentionally private. The rest of the project can stay user-facing because this notebook absorbs the messy details of notebooks as data structures.

The helpers here deliberately stay small because every higher-level tool depends on them. For example, `parse_cells` turns friendly cell-block text into notebook cells, `cell_hash` makes stale edits detectable, and the semantic classifiers let readers say "show me tests" or "show me exported code" without parsing raw notebook JSON.

In [ ]:
from contextlib import redirect_stdout as _redirect_stdout
from io import StringIO as _StringIO
import os as _os
from fastcore.nbio import mk_cell as _mk_cell, new_nb as _new_nb


In [ ]:
#| export
import ast
import hashlib
import json
import os
import re
import shutil
import subprocess
import sys
import time
import traceback
from contextlib import contextmanager
from functools import wraps
from pathlib import Path
from shutil import rmtree

from fastcore.nbio import mk_cell, new_nb
from fastcore.nbio import write_nb
from fastcore.script import _in_call_parse

### Demo scratch files

Examples and tests in later notebooks need small notebooks they can safely mutate. These helpers keep that setup in one place: they create named artifacts under nbs/data, reset them before use, and remove them when the example is finished.

In [ ]:
#| export
def remove_demo_path(path):
    path = Path(path)
    if path.is_dir():
        rmtree(path)
    elif path.exists():
        path.unlink()
    return path


def demo_path(name, base="nbs/data", reset=True):
    path = Path(base) / name
    if reset: remove_demo_path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    return path


def write_demo_notebook(name, cells=None, base="nbs/data", reset=True):
    path = demo_path(name, base=base, reset=reset)
    cells = cells or [
        mk_cell("## Demo notebook\nThis tiny notebook gives nbskill tools something real to inspect.", cell_type="markdown"),
        mk_cell("#| export\ndef demo_answer():\n    return 42"),
        mk_cell("assert demo_answer() == 42"),
    ]
    nb = stamp_notebook_metadata(new_nb(cells))
    write_nb(nb, path)
    return path

### CLI contracts and diagnostics

The public functions in later notebooks are both Python functions and command-line commands. These helpers make that dual use predictable: direct Python calls return values, CLI calls print user-friendly errors, and tool starts or failures are recorded in a small local failure map for debugging repeated friction.

In [ ]:
#| export
def cli_return(value=None):
    return None if _in_call_parse.get() else value

In [ ]:
#| export
def cli_error(msg):
    if _in_call_parse.get():
        print(msg, file=sys.stderr)
        raise SystemExit(1)
    raise ValueError(msg)

In [ ]:
#| export
def _failure_map_path():
    default = Path.home() / ".nbskill-errors.json"
    return Path(os.environ.get("NBSKILL_FAILURE_MAP", default)).expanduser()


In [ ]:
#| export
def _empty_failure_map():
    return {"version": 1, "events": [], "counts": {}, "last_call": None}


In [ ]:
#| export
def _load_failure_map(path):
    try:
        data = json.loads(path.read_text(encoding="utf-8"))
    except (FileNotFoundError, json.JSONDecodeError, OSError):
        data = _empty_failure_map()
    data.setdefault("version", 1)
    data.setdefault("events", [])
    data.setdefault("counts", {})
    data.setdefault("last_call", None)
    return data

In [ ]:
#| export
def _bump_count(data, kind, tool):
    counts = data.setdefault("counts", {})
    group = counts.setdefault(kind, {})
    group[tool] = group.get(tool, 0) + 1

In [ ]:
#| export
def _call_details(args, kwargs):
    details = {"cwd": str(Path.cwd())}
    if args and isinstance(args[0], (str, Path)): details["path"] = str(args[0])
    for key in ("path", "cell_id", "chapter", "source_hash"):
        value = kwargs.get(key)
        if value is not None: details[key] = str(value)
    return details

In [ ]:
#| export
def _write_failure_map(path, data):
    data["events"] = data.get("events", [])[-200:]
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=2, sort_keys=True), encoding="utf-8")

In [ ]:
#| export
def _record_tool_start(tool, details=None):
    path = _failure_map_path()
    now = time.time()
    details = details or {}
    event = {"tool": tool, "ts": now, **details}
    try:
        data = _load_failure_map(path)
        _bump_count(data, "usage", tool)
        last = data.get("last_call")
        if last:
            delta = now - float(last.get("ts", now))
            reasons = []
            if last.get("tool") == tool: reasons.append("same_tool")
            if delta <= 1.0: reasons.append("within_1s")
            if reasons:
                _bump_count(data, "friction", tool)
                data["events"].append({
                    "kind": "friction",
                    "tool": tool,
                    "path": details.get("path"),
                    "cell_id": details.get("cell_id"),
                    "previous_tool": last.get("tool"),
                    "previous_path": last.get("path"),
                    "seconds_since_previous": round(delta, 3),
                    "reasons": reasons,
                    "ts": now,
                })
        data["last_call"] = event
        _write_failure_map(path, data)
    except OSError:
        pass
    return event


In [ ]:
#| export
def _record_tool_failure(event, exc):
    path = _failure_map_path()
    try:
        data = _load_failure_map(path)
        tool = event["tool"]
        summary = "".join(traceback.format_exception_only(type(exc), exc)).strip()
        _bump_count(data, "failures", tool)
        data["events"].append({
            "kind": "failure",
            "tool": tool,
            "path": event.get("path"),
            "cell_id": event.get("cell_id"),
            "chapter": event.get("chapter"),
            "source_hash": event.get("source_hash"),
            "cwd": event.get("cwd"),
            "error_type": type(exc).__name__,
            "error": str(exc),
            "summary": summary,
            "ts": time.time(),
        })
        _write_failure_map(path, data)
    except OSError:
        pass

In [ ]:
#| export
@contextmanager
def _track_tool(tool, details=None):
    event = _record_tool_start(tool, details=details)
    try:
        yield
    except BaseException as exc:
        _record_tool_failure(event, exc)
        raise

In [ ]:
#| export
_NBSKILL_HOOKS_MARKER_START = "# nbskill nbdev hooks:start"
_NBSKILL_HOOKS_MARKER_END = "# nbskill nbdev hooks:end"
_NBSKILL_HOOK_ROOTS = set()


def _git_root(path="."):
    proc = subprocess.run(["git", "-C", str(path), "rev-parse", "--show-toplevel"], text=True, capture_output=True)
    return Path(proc.stdout.strip()) if proc.returncode == 0 and proc.stdout.strip() else None


def _looks_like_nbdev_project(root):
    root = Path(root)
    pyproject = root / "pyproject.toml"
    if (root / "nbs").exists() or (root / "settings.ini").exists(): return True
    return pyproject.exists() and "[tool.nbdev]" in pyproject.read_text(encoding="utf-8", errors="ignore")


def _nbdev_hook_block():
    return f"""{_NBSKILL_HOOKS_MARKER_START}
run_nbdev_cmd() {{
  if command -v "$1" >/dev/null 2>&1; then
    "$@"
  elif command -v uv >/dev/null 2>&1; then
    uv run "$@"
  else
    echo "nbskill: missing $1; install nbdev or uv" >&2
    exit 127
  fi
}}

run_nbdev_cmd nbdev-clean
if ! git diff --quiet -- .; then
  echo "nbskill: nbdev-clean changed notebooks. Review and stage those changes before committing." >&2
  exit 1
fi
run_nbdev_cmd nbdev-test
{_NBSKILL_HOOKS_MARKER_END}
"""


def _replace_marked_block(text, block):
    if _NBSKILL_HOOKS_MARKER_START in text and _NBSKILL_HOOKS_MARKER_END in text:
        before = text.split(_NBSKILL_HOOKS_MARKER_START, 1)[0].rstrip()
        after = text.split(_NBSKILL_HOOKS_MARKER_END, 1)[1].lstrip()
        return f"{before}\n\n{block}\n{after}".rstrip() + "\n"
    prefix = text.rstrip() if text.strip() else "#!/bin/sh"
    return f"{prefix}\n\n{block}\n"


def install_nbdev_pre_commit_hooks(path=".", run_nbdev_install_hooks=True):
    "Install nbdev-clean and nbdev-test pre-commit hooks in a git-backed nbdev project."
    root = _git_root(path)
    if root is None: return {"installed": False, "reason": "not-a-git-repo"}
    if not _looks_like_nbdev_project(root): return {"installed": False, "reason": "not-an-nbdev-project", "root": str(root)}
    if run_nbdev_install_hooks:
        cmd = ["nbdev-install-hooks"] if shutil.which("nbdev-install-hooks") else None
        if cmd is None and shutil.which("uv"): cmd = ["uv", "run", "nbdev-install-hooks"]
        if cmd is not None: subprocess.run(cmd, cwd=root, text=True, capture_output=True)
    hooks = root / ".git" / "hooks"
    hooks.mkdir(parents=True, exist_ok=True)
    pre_commit = hooks / "pre-commit"
    text = pre_commit.read_text(encoding="utf-8", errors="ignore") if pre_commit.exists() else ""
    pre_commit.write_text(_replace_marked_block(text, _nbdev_hook_block()), encoding="utf-8")
    pre_commit.chmod(pre_commit.stat().st_mode | 0o111)
    return {"installed": True, "root": str(root), "hook": str(pre_commit)}


def _ensure_nbdev_pre_commit_hooks(path="."):
    if os.environ.get("NBSKILL_NO_INSTALL_HOOKS"): return None
    root = _git_root(path)
    if root is None or str(root) in _NBSKILL_HOOK_ROOTS: return None
    _NBSKILL_HOOK_ROOTS.add(str(root))
    try: return install_nbdev_pre_commit_hooks(root)
    except BaseException: return None


def tracked_call(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        _ensure_nbdev_pre_commit_hooks()
        with _track_tool(func.__name__, details=_call_details(args, kwargs)):
            return func(*args, **kwargs)
    return wrapper

### Parsing user-facing selectors

Many tools accept inputs from shells, MCP clients, or notebooks, so values often arrive as strings. This section normalizes `None`, literal values, indexes, and slices before the higher-level tools try to select or edit cells.

In [ ]:
#| export
def parse_literal(value):
    if value is None: return None
    if isinstance(value, str):
        value = value.strip()
        if value.lower() in {"", "none", "null"}: return None
        try: return ast.literal_eval(value)
        except (SyntaxError, ValueError): return value
    return value

In [ ]:
#| export
def none_if_string(value):
    return None if isinstance(value, str) and value.strip().lower() in {"", "none", "null"} else value

In [ ]:
#| export
def _parse_slice(value):
    if not isinstance(value, str) or ":" not in value: return None
    parts = value.split(":")
    if len(parts) not in (2, 3): return None
    vals = [int(p) if p else None for p in parts]
    return slice(*vals)

In [ ]:
#| export
def _as_index(value, length):
    idx = int(value)
    if idx < 0: idx += length
    if idx < 0 or idx >= length: raise IndexError(value)
    return idx

In [ ]:
#| export
def _parse_read_selector(value):
    value = parse_literal(value)
    if value is None: return None
    if isinstance(value, str):
        slc = _parse_slice(value)
        if slc is not None: return slc
        return int(value)
    if isinstance(value, (list, tuple)): return [int(o) for o in value]
    return int(value)

In [ ]:
#| export
def _parse_write_target(value):
    value = parse_literal(value)
    if value is None: return None
    if isinstance(value, str):
        slc = _parse_slice(value)
        if slc is not None: return slc
        return int(value)
    if isinstance(value, (list, tuple)):
        if len(value) != 2: raise ValueError("write ranges must have start and stop")
        return slice(value[0], value[1])
    return int(value)

In [ ]:
#| export
def _select_cells(cells, selector):
    items = list(enumerate(cells))
    if selector is None: return items
    selector = _parse_read_selector(selector)
    if isinstance(selector, slice): return items[selector]
    if isinstance(selector, list):
        return [(idx, cells[idx]) for idx in (_as_index(o, len(cells)) for o in selector)]
    idx = _as_index(selector, len(cells))
    return [(idx, cells[idx])]

In [ ]:
#| export
def _delete_cells(cells, selector):
    if selector is None: return
    target = _parse_write_target(selector)
    if isinstance(target, slice):
        del cells[target]
        return
    idx = _as_index(target, len(cells))
    del cells[idx]

### Turning text into notebook cells

`write_nb` and `update_cell` accept plain text blocks instead of raw notebook JSON. These helpers split `---` separated blocks, honor `%%markdown` and `%%code` markers, and split large exported code cells into smaller symbol-sized cells when that makes edits safer.

In [ ]:
#| export
def _split_blocks(text):
    text = "" if text is None else str(text)
    if not text: return []
    return [o.strip("\n") for o in re.split(r"(?m)^\s*---\s*$", text) if o.strip()]

In [ ]:
#| export
def _coerce_cell(cell, default_type="code"):
    if isinstance(cell, dict): return cell
    if isinstance(cell, (tuple, list)) and len(cell) == 2:
        cell_type, source = cell
        return mk_cell(str(source), cell_type=str(cell_type))
    return mk_cell(str(cell), cell_type=default_type)

In [ ]:
#| export
def is_definition_node(node):
    return isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef))

In [ ]:
#| export
def node_start_line(node):
    return min([node.lineno, *[d.lineno for d in getattr(node, "decorator_list", [])]]) - 1

In [ ]:
#| export
def is_export_directive(line):
    return re.match(r"^\s*#\|\s*(export|exports|exporti)(\s|$)", line) is not None

In [ ]:
#| export
def _is_export_gap(lines):
    return bool(lines) and all((not line.strip()) or is_export_directive(line) for line in lines)

In [ ]:
#| export
def _emit_code_chunk(chunks, lines, export_prefix=None):
    if export_prefix: lines = [*export_prefix, *lines]
    text = "\n".join(lines).strip("\n")
    if text: chunks.append(text)

In [ ]:
#| export
def _split_code_cell_sources(source):
    source = source.strip("\n")
    if not source: return []
    try: tree = ast.parse(source)
    except SyntaxError: return [source]
    if sum(1 for node in tree.body if is_definition_node(node)) <= 1: return [source]

    lines = source.splitlines()
    first_start = node_start_line(tree.body[0]) if tree.body else 0
    leading = lines[:first_start]
    shared_export = [line for line in leading if is_export_directive(line)] if _is_export_gap(leading) else []
    chunks = []
    if shared_export:
        cursor = first_start
    else:
        _emit_code_chunk(chunks, leading)
        cursor = first_start

    for node in tree.body:
        start = node_start_line(node)
        end = node.end_lineno
        gap = lines[cursor:start]
        if shared_export and _is_export_gap(gap): gap = []
        _emit_code_chunk(chunks, [*gap, *lines[start:end]], shared_export or None)
        cursor = end
    _emit_code_chunk(chunks, lines[cursor:])
    return chunks or [source]

In [ ]:
#| export
def _split_code_cell(cell):
    cell_type = cell.get("cell_type") if isinstance(cell, dict) else getattr(cell, "cell_type", None)
    if cell_type != "code": return [cell]
    source = cell.get("source", "") if isinstance(cell, dict) else getattr(cell, "source", "")
    if isinstance(source, list): source = "".join(source)
    sources = _split_code_cell_sources(str(source))
    if len(sources) <= 1: return [cell]
    return [mk_cell(source, cell_type="code") for source in sources]

In [ ]:
#| export
def _split_symbol_cells(cells):
    split = []
    for cell in cells: split.extend(_split_code_cell(cell))
    return split

In [ ]:
#| export
def cell_source(cell):
    source = cell.get("source", "") if isinstance(cell, dict) else getattr(cell, "source", "")
    if isinstance(source, list): return "".join(source)
    return str(source)

In [ ]:
#| export
def cell_hash(cell_or_source, n=12):
    source = cell_source(cell_or_source) if not isinstance(cell_or_source, str) else cell_or_source
    digest = hashlib.sha256(source.encode("utf-8")).hexdigest()
    return digest if n is None else digest[:n]

### Cell metadata and semantic classes

Notebook tools repeatedly ask questions like "is this an exported cell?" or "is this a test cell?" The metadata helpers keep a place for cached semantic information while still falling back to computing classes from the live source when needed.

In [ ]:
#| export
_NBSKILL_METADATA_KEY = "nbskill"


def cell_metadata(cell):
    meta = cell.get("metadata", None) if isinstance(cell, dict) else getattr(cell, "metadata", None)
    if meta is None:
        meta = {}
        if isinstance(cell, dict): cell["metadata"] = meta
        else: cell.metadata = meta
    return meta


def notebook_metadata(nb):
    meta = nb.get("metadata", None) if isinstance(nb, dict) else getattr(nb, "metadata", None)
    if meta is None:
        meta = {}
        if isinstance(nb, dict): nb["metadata"] = meta
        else: nb.metadata = meta
    return meta


def _nbskill_cell_metadata(cell, create=True):
    meta = cell_metadata(cell) if create else (cell.get("metadata", {}) if isinstance(cell, dict) else getattr(cell, "metadata", {}) or {})
    info = meta.get(_NBSKILL_METADATA_KEY) if isinstance(meta, dict) else None
    if isinstance(info, dict): return info
    if not create: return None
    info = {}
    meta[_NBSKILL_METADATA_KEY] = info
    return info


def _nbskill_notebook_metadata(nb, create=True):
    meta = notebook_metadata(nb) if create else (nb.get("metadata", {}) if isinstance(nb, dict) else getattr(nb, "metadata", {}) or {})
    info = meta.get(_NBSKILL_METADATA_KEY) if isinstance(meta, dict) else None
    if isinstance(info, dict): return info
    if not create: return None
    info = {}
    meta[_NBSKILL_METADATA_KEY] = info
    return info


def file_hash(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def _metadata_path(path):
    path = Path(path)
    try: return path.resolve().relative_to(Path.cwd().resolve()).as_posix()
    except (OSError, ValueError): return path.as_posix()


def _default_exp_from_notebook(nb):
    for cell in getattr(nb, "cells", []):
        for line in cell_source(cell).splitlines():
            match = re.match(r"^\s*#\|\s*default_exp\s+(.+?)\s*$", line)
            if match: return match.group(1).strip()
    return None


def exported_py_path(nb_path, nb=None):
    "Return the generated Python file path for an nbdev notebook, if it has one."
    nb_path = Path(nb_path)
    if nb is None:
        from fastcore.nbio import read_nb as _read_nb
        nb = _read_nb(nb_path)
    default_exp = _default_exp_from_notebook(nb)
    if not default_exp: return None
    try:
        from nbdev.config import get_config
        lib_path = Path(get_config(nb_path.parent).lib_path)
    except Exception:
        lib_path = nb_path.parent.parent / default_exp.split(".", 1)[0]
    return lib_path / (default_exp.replace(".", "/") + ".py")


def stamp_export_metadata(nb, py_path):
    info = _nbskill_notebook_metadata(nb)
    info["exported_py_path"] = _metadata_path(py_path)
    info["exported_py_hash"] = file_hash(py_path)
    return nb


def _fresh_semantic_metadata(cell):
    info = _nbskill_cell_metadata(cell, create=False)
    if not info: return None
    if info.get("source_hash") != cell_hash(cell, n=None): return None
    if info.get("cell_type") != getattr(cell, "cell_type", None): return None
    types = info.get("semantic_types")
    if not isinstance(types, list): return None
    normalized = tuple("example_cell" if str(item) == "exploration_cell" else str(item) for item in types)
    if getattr(cell, "cell_type", None) != "code" and "unclean_cell" in normalized: return None
    return normalized

In [ ]:
#| export
def parse_one_cell(text, default_type="code"):
    cells = parse_cells(text, default_type)
    if len(cells) != 1: cli_error("update_cell expects exactly one replacement cell")
    return cells[0]

In [ ]:
#| export
def cell_matches_hash(cell, source_hash):
    if source_hash is None: return True
    return cell_hash(cell, n=None).startswith(str(source_hash).lower())

In [ ]:
#| export
def find_cell_by_id(cells, cell_id):
    matches = [(idx, cell) for idx, cell in enumerate(cells) if getattr(cell, "id", None) == cell_id]
    if len(matches) == 1: return matches[0]
    if not matches: cli_error(f"No cell has id {cell_id!r}")
    cli_error(f"Multiple cells have id {cell_id!r}")

In [ ]:
#| export
def find_cell_by_text(cells, old_str):
    matches = [(idx, cell) for idx, cell in enumerate(cells) if old_str in cell_source(cell)]
    if len(matches) == 1: return matches[0]
    if not matches: cli_error("old_str did not match any cell")
    idxs = ", ".join(str(idx) for idx, _ in matches)
    cli_error(f"old_str matched multiple cells: {idxs}. Use --cell_id or a more specific old_str.")

In [ ]:
#| export
def replace_cell(nb, idx, new_cell):
    old_id = getattr(nb.cells[idx], "id", None)
    if old_id is not None: new_cell.id = old_id
    nb.cells[idx] = new_cell

In [ ]:
#| export
def clear_outputs(cell):
    if getattr(cell, "cell_type", None) == "code":
        cell.outputs = []
        cell.execution_count = None
    return cell

In [ ]:
#| export
def _looks_like_multiline_cli_text(text):
    if not isinstance(text, str) or "\\n" not in text: return False
    stripped = text.lstrip().lower()
    if stripped.startswith(("%%code\\n", "%%markdown\\n", "%%md\\n", "%%raw\\n")): return True
    if "\\n---\\n" in text: return True
    return "\\n    " in text or "\\n\t" in text


def _decode_cli_newlines(text):
    return text.replace("\\n", "\n") if _looks_like_multiline_cli_text(text) else text


def load_cells_text(cells="", cells_file=None):
    if cells_file:
        if cells: raise ValueError("Use either cells or cells_file, not both")
        return Path(cells_file).expanduser().read_text(encoding="utf-8")
    if cells == "-": return sys.stdin.read()
    return _decode_cli_newlines(cells)


In [ ]:
#| export
def _should_validate_python(source):
    for line in source.splitlines():
        stripped = line.lstrip()
        if stripped.startswith(("%", "!")): return False
    return bool(source.strip())

In [ ]:
#| export
def _format_syntax_error(source, err, cell_idx):
    lines = source.splitlines()
    line = lines[err.lineno - 1] if err.lineno and 0 < err.lineno <= len(lines) else ""
    pointer = " " * max((err.offset or 1) - 1, 0) + "^" if line else ""
    msg = [f"Invalid Python in new code cell {cell_idx}: {err.msg} at line {err.lineno}, column {err.offset}"]
    if line: msg += [line, pointer]
    msg.append("Tip: shell quoting can turn backslash-n escapes into real newlines inside Python strings. Use --cells_file PATH or cells=- for complex code.")
    return chr(10).join(msg)

In [ ]:
#| export
def validate_code_cells(cells):
    for idx, cell in enumerate(cells):
        cell_type = cell.get("cell_type") if isinstance(cell, dict) else getattr(cell, "cell_type", None)
        if cell_type != "code": continue
        source = cell_source(cell)
        if not _should_validate_python(source): continue
        try: ast.parse(source)
        except SyntaxError as err:
            msg = _format_syntax_error(source, err, idx)
            if _in_call_parse.get(): raise SystemExit(msg)
            raise ValueError(msg) from err

In [ ]:
#| export
def parse_cells(cells, default_type="code"):
    if isinstance(cells, (list, tuple)): return _split_symbol_cells([_coerce_cell(o, default_type) for o in cells])

    parsed = []
    for block in _split_blocks(cells):
        lines = block.splitlines()
        marker = lines[0].strip().lower() if lines else ""
        cell_type = default_type
        if marker in {"%%markdown", "%%md"}:
            cell_type, lines = "markdown", lines[1:]
        elif marker == "%%code":
            cell_type, lines = "code", lines[1:]
        elif marker == "%%raw":
            cell_type, lines = "raw", lines[1:]
        parsed.append(mk_cell("\n".join(lines), cell_type=cell_type))
    return _split_symbol_cells(parsed)

### Naming cells by behavior

The reading and MCP layers present cells by meaning, not just by `code` or `markdown`. This section detects imports, private helpers, exported code, tests, examples, docs, section headers, and mixed cells so callers can filter notebooks at a useful level.

In [ ]:
#| export
def first_line(source):
    for line in source.splitlines():
        line = line.strip()
        if line: return line
    return ""

In [ ]:
#| export
def cell_prefix(idx, cell, show_ids=False):
    suffix = f" hash={cell_hash(cell)}" if show_ids else ""
    classes = cell_class_names(cell)
    class_suffix = f" classes={','.join(classes)}" if classes else ""
    return f"Cell id={cell.id}{suffix}: {cell.cell_type}{class_suffix}"

In [ ]:
#| export
def _format_chapter_spans(spans, cells, show_ids=False):
    lines = []
    for span in spans:
        cell = cells[span["start"]]
        suffix = f" hash={cell_hash(cell)}" if show_ids else ""
        lines.append(f"Chapter id={cell.id}{suffix}: ## {span['title']}")
    return "\n".join(lines)

In [ ]:
#| export
def matches_filter(source, pattern):
    pattern = str(pattern)
    if pattern in source: return True
    try: return re.search(pattern, source, flags=re.MULTILINE) is not None
    except re.error: return False

In [ ]:
#| export
def is_exported_code_cell(cell):
    if getattr(cell, "cell_type", None) != "code": return False
    return any(is_export_directive(line) for line in cell_source(cell).splitlines())


def _cell_outputs(cell):
    return list(getattr(cell, "outputs", []) or [])


def _has_cell_output(cell):
    return bool(_cell_outputs(cell))


def _code_tree(cell):
    try: return ast.parse(cell_source(cell))
    except SyntaxError: return None


def _code_body(cell):
    tree = _code_tree(cell)
    return [] if tree is None else list(tree.body)


def _is_import_cell(cell):
    if getattr(cell, "cell_type", None) != "code": return False
    body = _code_body(cell)
    return bool(body) and all(isinstance(node, (ast.Import, ast.ImportFrom)) for node in body)


def _has_private_function(cell):
    if getattr(cell, "cell_type", None) != "code": return False
    tree = _code_tree(cell)
    if tree is None: return False
    return any(
        isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name.startswith("_")
        for node in tree.body
    )


def _has_test_marker(cell):
    source = cell_source(cell)
    tree = _code_tree(cell)
    if tree is None: return re.search(r"\b(assert|test_[A-Za-z0-9_]*)\b", source) is not None
    if any(isinstance(node, ast.Assert) for node in ast.walk(tree)): return True
    return any(
        isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name.startswith("test_")
        for node in tree.body
    )


def _is_test_cell(cell):
    return getattr(cell, "cell_type", None) == "code" and not _has_cell_output(cell) and _has_test_marker(cell)


def _is_example_cell(cell):
    return getattr(cell, "cell_type", None) == "code" and _has_cell_output(cell)


def _is_section_header(cell):
    if getattr(cell, "cell_type", None) != "markdown": return False
    return any(re.match(r"^#{1,2}\s+", line.strip()) for line in cell_source(cell).splitlines())


def _is_docs_cell(cell):
    if getattr(cell, "cell_type", None) != "markdown": return False
    return any(line.strip() and not re.match(r"^#{1,2}\s+", line.strip()) for line in cell_source(cell).splitlines())


def _cell_base_class_names(cell):
    names = []
    if _is_import_cell(cell): names.append("import_cell")
    if _is_example_cell(cell): names.append("example_cell")
    if _is_test_cell(cell): names.append("test_cell")
    if _has_private_function(cell): names.append("private_code")
    if is_exported_code_cell(cell): names.append("exported_code")
    if _is_docs_cell(cell): names.append("docs_cell")
    if _is_section_header(cell): names.append("section_header")
    return tuple(names)


def _semantic_code_class_names(cell):
    semantic = {"import_cell", "example_cell", "test_cell", "private_code", "exported_code"}
    return tuple(name for name in _cell_base_class_names(cell) if name in semantic)


def _fallback_cell_class_name(cell):
    cell_type = getattr(cell, "cell_type", None)
    return f"{cell_type}_cell" if cell_type else "unknown_cell"


def _computed_cell_class_names(cell):
    names = _cell_base_class_names(cell)
    if getattr(cell, "cell_type", None) == "code" and len(_semantic_code_class_names(cell)) > 1:
        return (*names, "unclean_cell")
    return names or (_fallback_cell_class_name(cell),)


def _refresh_cell_metadata(cell):
    info = _nbskill_cell_metadata(cell)
    semantic_types = _computed_cell_class_names(cell)
    info["cell_type"] = getattr(cell, "cell_type", None)
    info["semantic_types"] = list(semantic_types)
    info["source_hash"] = cell_hash(cell, n=None)
    return cell


def stamp_notebook_metadata(nb, exported_py_path=None):
    for cell in getattr(nb, "cells", []): _refresh_cell_metadata(cell)
    if exported_py_path is not None: stamp_export_metadata(nb, exported_py_path)
    return nb


def cell_class_names(cell):
    return _fresh_semantic_metadata(cell) or _computed_cell_class_names(cell)


In [ ]:
_demo_cells = parse_cells("%%markdown\n## Demo\n---\n%%code\nvalue = 42")
for idx, cell in enumerate(_demo_cells):
    print(cell_prefix(idx, cell, show_ids=True))
print("code hash:", cell_hash(_demo_cells[1]))

Cell id=e408ff00 hash=876a02aca812: markdown classes=section_header
Cell id=276966fe hash=d6a0e509ba31: code
code hash: d6a0e509ba31


In [ ]:
#| export
def _normalize_cell_type_filter(value):
    if value is None: return None
    aliases = {
        "code": "code",
        "py": "code",
        "python": "code",
        "md": "markdown",
        "markdown": "markdown",
        "doc": "markdown",
        "docs": "markdown",
        "raw": "raw",
        "export": "exported_code",
        "exported": "exported_code",
        "exported_code": "exported_code",
        "import": "import_cell",
        "imports": "import_cell",
        "import_cell": "import_cell",
        "private": "private_code",
        "private_code": "private_code",
        "test": "test_cell",
        "tests": "test_cell",
        "test_cell": "test_cell",
        "example": "example_cell",
        "examples": "example_cell",
        "example_cell": "example_cell",
        "exploration": "example_cell",
        "explorations": "example_cell",
        "exploration_cell": "example_cell",
        "docs_cell": "docs_cell",
        "documentation": "docs_cell",
        "section": "section_header",
        "header": "section_header",
        "section_header": "section_header",
        "unclean": "unclean_cell",
        "unclean_cell": "unclean_cell",
    }
    normalized = set()
    for item in str(value).split(","):
        key = item.strip().lower()
        if not key: continue
        if key not in aliases:
            choices = ", ".join(sorted(set(aliases)))
            raise ValueError(f"Unknown cell_type {item!r}; use one of: {choices}")
        normalized.add(aliases[key])
    return normalized or None


def cell_matches_type(cell, cell_type):
    wanted = _normalize_cell_type_filter(cell_type)
    if wanted is None: return True
    if getattr(cell, "cell_type", None) in wanted: return True
    return bool(set(cell_class_names(cell)) & wanted)


def with_context(cells, items, include=False):
    if not include: return items

    idxs = {idx for idx, _ in items}
    for idx in list(idxs):
        prev = idx - 1
        while prev >= 0 and cells[prev].cell_type == "markdown":
            idxs.add(prev)
            prev -= 1

        nxt = idx + 1
        while nxt < len(cells):
            cell = cells[nxt]
            if cell.cell_type != "code" or is_exported_code_cell(cell): break
            idxs.add(nxt)
            nxt += 1
    return [(idx, cells[idx]) for idx in sorted(idxs)]


### Chapters as editing scopes

A chapter is a Markdown `##` section plus the cells that follow it. Chapter helpers let tools target a named section, create one if requested, or insert into a section without counting global cell indexes by hand.

In [ ]:
#| export
def _chapter_title(cell):
    if getattr(cell, "cell_type", None) != "markdown": return None
    for line in cell_source(cell).splitlines():
        match = re.match(r"^##\s+(.+?)\s*$", line.strip())
        if match: return match.group(1).strip()
    return None

In [ ]:
#| export
def _chapter_spans(cells):
    starts = [(idx, title) for idx, cell in enumerate(cells) if (title := _chapter_title(cell))]
    spans = []
    for pos, (start, title) in enumerate(starts):
        end = starts[pos + 1][0] if pos + 1 < len(starts) else len(cells)
        spans.append(dict(title=title, start=start, end=end))
    return spans

In [ ]:
#| export
def _matching_chapters(cells, chapter=None):
    spans = _chapter_spans(cells)
    if chapter is None: return spans
    return [span for span in spans if matches_filter(span["title"], chapter)]

In [ ]:
#| export
def chapter_index_set(cells, chapter):
    idxs = set()
    for span in _matching_chapters(cells, chapter):
        idxs.update(range(span["start"], span["end"]))
    return idxs

In [ ]:
#| export
def one_chapter(cells, chapter, create=False):
    matches = _matching_chapters(cells, chapter)
    if len(matches) == 1: return matches[0]
    if not matches and create:
        cells.append(mk_cell(f"## {chapter}", cell_type="markdown"))
        return dict(title=str(chapter), start=len(cells) - 1, end=len(cells))
    if not matches: raise ValueError(f"No chapter matches {chapter!r}")
    titles = ", ".join(f"{span['title']} ({span['start']}:{span['end']})" for span in matches)
    raise ValueError(f"Chapter {chapter!r} matches multiple chapters: {titles}")

In [ ]:
#| export
def _chapter_body_len(span):
    return max(span["end"] - span["start"] - 1, 0)

In [ ]:
#| export
def _chapter_body_slice(span, target):
    body_start = span["start"] + 1
    body_len = _chapter_body_len(span)
    start, stop, step = target.indices(body_len)
    if step != 1: raise ValueError("chapter ranges do not support steps")
    return slice(body_start + start, body_start + stop)

In [ ]:
#| export
def _chapter_delete(cells, span, selector):
    if selector is None: return
    target = _parse_write_target(selector)
    body_start = span["start"] + 1
    body_len = _chapter_body_len(span)
    if isinstance(target, slice):
        del cells[_chapter_body_slice(span, target)]
        return
    idx = int(target)
    if idx < 0: idx += body_len
    if idx < 0 or idx >= body_len: raise IndexError(target)
    del cells[body_start + idx]

In [ ]:
#| export
def _chapter_insert_target(span, target):
    body_start = span["start"] + 1
    body_len = _chapter_body_len(span)
    if target is None: return slice(body_start, body_start + body_len)
    if isinstance(target, slice): return _chapter_body_slice(span, target)
    idx = int(target)
    if idx == -1: return body_start + body_len
    if idx < 0: idx += body_len
    if idx < 0 or idx > body_len: raise IndexError(target)
    return body_start + idx

In [ ]:
custom_map = demo_path("00_foundation_errors.json")
_old_failure_map = _os.environ.get("NBSKILL_FAILURE_MAP")
try:
    _os.environ["NBSKILL_FAILURE_MAP"] = str(custom_map)
    assert _failure_map_path() == custom_map
finally:
    if _old_failure_map is None:
        _os.environ.pop("NBSKILL_FAILURE_MAP", None)
    else:
        _os.environ["NBSKILL_FAILURE_MAP"] = _old_failure_map

cells = parse_cells("%%markdown\n## Demo\n---\n%%code\nvalue = 42")
assert len(cells) == 2
assert cells[0].cell_type == "markdown"
assert cells[1].source == "value = 42"
one = parse_one_cell("%%code\nvalue = 99")
assert one.cell_type == "code"
assert one.source == "value = 99"
assert load_cells_text("%%code\\nvalue = 1") == "%%code\nvalue = 1"
assert load_cells_text("def f():\\n    return 1") == "def f():\n    return 1"
assert load_cells_text("value = 'a\\nb'") == "value = 'a\\nb'"


In [ ]:
from fastcore.nbio import mk_cell as _mk_cell, new_nb as _new_nb
from nbskill.foundation import (
    cell_class_names, cell_matches_type, file_hash, notebook_metadata,
    remove_demo_path, stamp_notebook_metadata,
)

import_cell = _mk_cell("import os", cell_type="code")
private_cell = _mk_cell("""def _helper():
    pass""", cell_type="code")
exported_cell = _mk_cell("""#| export
def public():
    pass""", cell_type="code")
test_cell = _mk_cell("assert 1 == 1", cell_type="code")
plain_code_cell = _mk_cell("value = 1", cell_type="code")
example_cell = _mk_cell("value = 1", cell_type="code")
example_cell.outputs = [{"output_type": "stream", "name": "stdout", "text": "1\n"}]
docs_cell = _mk_cell("Some notes", cell_type="markdown")
section_cell = _mk_cell("## API", cell_type="markdown")
docs_header_cell = _mk_cell("""## API
Some notes""", cell_type="markdown")
unclean_cell = _mk_cell("""#| export
import os""", cell_type="code")

assert cell_class_names(import_cell) == ("import_cell",)
assert cell_class_names(private_cell) == ("private_code",)
assert cell_class_names(exported_cell) == ("exported_code",)
assert cell_class_names(test_cell) == ("test_cell",)
assert cell_class_names(plain_code_cell) == ("code_cell",)
assert cell_class_names(example_cell) == ("example_cell",)
assert cell_class_names(docs_cell) == ("docs_cell",)
assert cell_class_names(section_cell) == ("section_header",)
assert cell_class_names(docs_header_cell) == ("docs_cell", "section_header")
assert cell_class_names(unclean_cell) == ("import_cell", "exported_code", "unclean_cell")
assert cell_matches_type(example_cell, "example")
assert cell_matches_type(example_cell, "exploration")

export_path = demo_path("00_foundation_export.py")
export_path.write_text("""print('exported')
""", encoding="utf-8")
stamped_nb = stamp_notebook_metadata(_new_nb([exported_cell]), exported_py_path=export_path)
stamped_cell = stamped_nb.cells[0]
cell_info = stamped_cell.metadata["nbskill"]
notebook_info = notebook_metadata(stamped_nb)["nbskill"]
assert cell_info["cell_type"] == "code"
assert isinstance(cell_info["semantic_types"], list)
assert cell_info["source_hash"]
assert notebook_info["exported_py_hash"] == file_hash(export_path)
assert notebook_info["exported_py_path"].endswith("00_foundation_export.py")
assert cell_class_names(stamped_cell) == ("exported_code",)
stamped_cell.metadata["nbskill"]["semantic_types"] = ["stored_type"]
assert cell_class_names(stamped_cell) == ("stored_type",)
stamped_cell.source += chr(10)
assert cell_class_names(stamped_cell) == ("exported_code",)
remove_demo_path(export_path)